In [0]:
%run /Workspace/utilities

In [0]:
from datetime import *
from pyspark.sql.functions import current_timestamp

In [0]:
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','csv')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .option('cloudFiles.schemaLocation','/Volumes/projectdev/bronze/plane_schema/')\
    .load(f'abfss://raw@datalakestrgdev.dfs.core.windows.net/PLANE/Datepart={date.today()}/')

In [0]:
df_base = df.withColumnRenamed('tailnum','tailId')\
            .withColumn('last_load_ts',current_timestamp())

from delta.tables import DeltaTable
from pyspark.sql.functions import lit

target_table = 'projectdev.cleansed.plane'
def upsert_to_delta(microBatchDF, batchId):
    delta_table = DeltaTable.forName(spark,target_table)
    if not spark.catalog.tableExists(target_table):
        microBatchDF.write.mode("overwrite").format("delta").saveAsTable(target_table)
    else:
        last_watermark = spark.sql("""
                                   select coalesce(max(last_load_ts),'1900-01-01' ) from {target_table}
                                   """).collect()[0][0]
        
        incremental_df = df_base.filter(col("last_load_ts") > lit(last_watermark))\
                                .dropDuplicates(["iata_code"])
        (
            delta_table.alias("t")
            .merge(
                microBatchDF.alias("s"),
                "t.tailId = s.tailId"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

query = (
    df_base.writeStream
        .trigger(availableNow =True)
        .option('overwriteSchema','true')
        .foreachBatch(upsert_to_delta)
        .option("checkpointLocation", '/Volumes/projectdev/bronze/plane_schema/checkpoint/')
        .start()
)

query.awaitTermination()

# df_base.writeStream.trigger(once=True)\
#     .format('delta')\
#     .option("mergeSchema", "true")\
#     .outputMode('append')\
#     .option('checkpointLocation','/Volumes/projectdev/bronze/plane_schema/checkpoint/')\
#     .start('abfss://cleansed@datalakestrgdev.dfs.core.windows.net/plane')
     

In [0]:
%sql
create table if not exists projectdev.cleansed.plane
using delta
location 'abfss://cleansed@datalakestrgdev.dfs.core.windows.net/plane'